# BBC News Tech Subcategory Classifier

This notebook implements the full subcategory classification pipeline for 
the Tech category of the BBC News dataset.

## Pipeline Overview

- Phase 1 — Data Ingestion
- Phase 2 — Evidence Based Text Cleaning
- Phase 3 — Vectorization (Jina Embeddings v5)
- Phase 4 — UMAP Dimensionality Reduction and HDBSCAN Clustering
- Phase 5 — c-TF-IDF Thematic Labelling
- Phase 5B — KeyBERT Keyword Extraction and Cluster Verification
- Phase 5C — Post KeyBERT Flagged Cluster Review
- Phase 6 — AI Subcategory Naming and Final Output
- Phase 7 — Manual Noise Correction
- Phase 8A — Named Entity Recognition
- Phase 8B — April Events Extraction
- Phase 8C — Save Complete NER Output
- Phase 8D — Role Classifier Fix V1
- Phase 8E — Role Classifier Fix V2
- Phase 8F — Role Classifier Fix V3
- Final — Project Summary

## Project Overview

This notebook classifies 401 BBC Tech articles into meaningful subcategories. 
A diagnostic analysis was run prior to data cleaning to ensure only necessary 
cleaning steps were applied. Semantic embeddings are generated using Jina 
Embeddings v5, reduced to 3 dimensions using UMAP, and clustered using 
HDBSCAN density based clustering. Beyond subcategory classification the 
pipeline also performs Named Entity Recognition to extract persons, 
organisations and locations, assigns contextual roles to identified persons 
using three iterative classifier versions, and identifies events that took 
place or were scheduled in April.

## Dataset Details

- Author: BBC NLP Project 2
- Dataset: 401 BBC Tech Articles
- Model: Jina Embeddings v5 text-small with clustering adapter
- Subcategories Found: 22
- Noise Articles: 0
- Pipeline Phases: 16


## Dataset Details

- Author: BBC NLP Project 2
- Dataset: 401 BBC Tech Articles
- Model: Jina Embeddings v5 text-small with clustering adapter
- Subcategories Found: 22
- Noise Articles: 0
- Pipeline Phases: 10
---

## Phase 1: Data Ingestion & Structural Partitioning
In this phase we load all 401 tech articles from the `./tech` folder into a structured DataFrame. Each article is split into its title and body, and metadata is tracked alongside the raw text.

In [1]:
# ============================================================
# PHASE 1: DATA INGESTION & STRUCTURAL PARTITIONING
# ============================================================

import os
import pandas as pd

# 1. Define the path to the tech folder
folder_path = './tech'

# 2. Initialize our data collection list
all_articles = []

# 3. Loop through each .txt file in the tech folder
for filename in sorted(os.listdir(folder_path)):
    if filename.endswith('.txt'):
        file_path = os.path.join(folder_path, filename)

        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read().strip()

        # 4. Split the file into title (first line) and body (rest)
        lines = content.split('\n')
        title = lines[0].strip()
        body = '\n'.join(lines[1:]).strip()

        # 5. Track metadata alongside raw text
        all_articles.append({
            'file_name': filename,
            'article_id': int(filename.replace('.txt', '')),
            'title': title,
            'raw_text': body,
            'article_length': len(body.split())
        })

# 6. Load into a structured DataFrame
df = pd.DataFrame(all_articles)

# 7. Quick sanity check
print(f" Total articles loaded: {len(df)}")
print(f" Columns: {df.columns.tolist()}")
print(f"\n Sample:")
df.head()

 Total articles loaded: 401
 Columns: ['file_name', 'article_id', 'title', 'raw_text', 'article_length']

 Sample:


,file_name,article_id,title,raw_text,article_length
0,001.txt,1,Ink helps drive democracy in Asia,"The Kyrgyz Republic, a small, mountainous stat...",666
1,002.txt,2,China net cafe culture crackdown,"Chinese authorities closed 12,575 net cafes in...",378
2,003.txt,3,Microsoft seeking spyware trojan,Microsoft is investigating a trojan program th...,208
3,004.txt,4,Digital guru floats sub-$100 PC,"Nicholas Negroponte, chairman and founder of M...",459
4,005.txt,5,Technology gets the creative bug,The hi-tech and the arts worlds have for some ...,799


## Phase 2: The Refinement Layer (Data Cleaning)

In this phase we clean all tech articles to prepare them for vectorization.
The cleaning process removes HTML tags and normalises whitespace to ensure
the text is consistent and ready for embedding.

In [2]:

# ============================================================
# PHASE 2: DATA CLEANING
# ============================================================
import re
import pandas as pd

def clean_article(text):

    # 1. HTML REMOVAL - confirmed present in raw BBC data
    text = re.sub(r'<[^>]+>', '', text)

    # 2. WHITESPACE CLEANUP
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()

    return text

df = pd.read_csv('tech_subcategories_output.csv')

if 'raw_text' not in df.columns:
    df = df.rename(columns={'cleaned_text': 'raw_text'})

df['cleaned_text'] = df['raw_text'].apply(clean_article)
df = df.drop(columns=['raw_text'])

df.to_csv('tech_subcategories_output.csv', index=False)

print(f"Total articles: {len(df)}")
print(f"\nTech cleaning complete!")

Total articles: 401

Tech cleaning complete!


## Phase 3: Vectorization Layer (Jina-v5 Intelligence)
In this phase we convert each cleaned article into a 1024-dimensional embedding vector using Jina AI's latest embedding model with the clustering adapter. Articles are processed in batches of 20 with auto-retry on rate limit errors.

In [3]:
# ============================================================
# PHASE 3: THE VECTORIZATION LAYER (JINA-V5) - RATE LIMIT FIX
# ============================================================

import requests
import numpy as np
import time
import os

JINA_API_KEY = "jina_0630e4e341144046ad7def8c563d0a7dcCRoxZqTtbVC15To_XeVA2-9PnKx"
JINA_URL = "https://api.jina.ai/v1/embeddings"

HEADERS = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {JINA_API_KEY}"
}

def get_embeddings(texts, batch_size=20):
    all_embeddings = []
    total_batches = (len(texts) + batch_size - 1) // batch_size

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        batch_num = (i // batch_size) + 1

        payload = {
            "model": "jina-embeddings-v5-text-small",
            "task": "text-matching",
            "normalized": True,
            "dimensions": 1024,
            "input": batch
        }

        success = False
        retries = 3

        while not success and retries > 0:
            try:
                response = requests.post(JINA_URL, headers=HEADERS, json=payload)
                response.raise_for_status()
                data = response.json()

                batch_embeddings = [item['embedding'] for item in data['data']]
                all_embeddings.extend(batch_embeddings)

                print(f" Batch {batch_num}/{total_batches} done — {len(all_embeddings)} articles embedded so far")
                success = True

                time.sleep(3)

            except Exception as e:
                retries -= 1
                print(f" Rate limit hit on batch {batch_num}, retrying in 10s... ({retries} retries left)")
                time.sleep(10)

        if not success:
            print(f" Failed on batch {batch_num} after all retries")
            break

    return np.array(all_embeddings)

# Only embed if embeddings don't already exist
if os.path.exists('tech_embeddings.npy'):
    print(" Embeddings already exist — skipping API call")
    embeddings = np.load('tech_embeddings.npy')
else:
    # Run on ALL 401 articles using title + content combined
    texts = (df['title'] + " " + df['cleaned_text']).tolist()
    print(f" Embedding {len(texts)} articles with Jina-v5 clustering adapter...\n")

    embeddings = get_embeddings(texts)

    # Save embeddings to disk permanently
    np.save('tech_embeddings.npy', embeddings)
    print(f"\n Embedding complete!")

print(f" Embedding matrix shape: {embeddings.shape}")
print(f"   → {embeddings.shape[0]} articles x {embeddings.shape[1]} dimensions")

 Embeddings already exist — skipping API call
 Embedding matrix shape: (401, 1024)
   → 401 articles x 1024 dimensions


## Phase 4: Dimensionality Reduction & Clustering
In this phase we reduce the 1024-dimensional embeddings to 3 dimensions using UMAP while preserving thematic relationships between articles. HDBSCAN then discovers the natural number of subcategories automatically without any pre-set cluster count. Results are saved to disk permanently to ensure reproducibility.

In [4]:
# ============================================================
# PHASE 4: DIMENSIONALITY REDUCTION & CLUSTERING
# ============================================================

import umap
import hdbscan
import numpy as np
import os

# Load embeddings from disk if available
if os.path.exists('tech_embeddings.npy'):
    print(" Loading saved embeddings...")
    embeddings = np.load('tech_embeddings.npy')
    print(f" Embeddings shape: {embeddings.shape}")

# Check if we already have saved cluster results
if os.path.exists('tech_cluster_labels.npy') and os.path.exists('tech_reduced_embeddings.npy'):
    
    print(" Loading saved cluster results...")
    cluster_labels = np.load('tech_cluster_labels.npy')
    reduced_embeddings = np.load('tech_reduced_embeddings.npy')
    confidence_scores = np.load('tech_confidence_scores.npy')
    
    df['subcategory_id'] = cluster_labels
    df['confidence_score'] = confidence_scores

    n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
    n_noise = list(cluster_labels).count(-1)

    print(f" Subcategories found: {n_clusters}")
    print(f"  Noise articles: {n_noise}")
    print(f" Clustered articles: {len(cluster_labels) - n_noise}")

else:
    
    # 1. UMAP
    print(" Running UMAP dimensionality reduction...")
    reducer = umap.UMAP(
        n_components=3,
        n_neighbors=15,
        min_dist=0.0,
        metric='cosine',
        random_state=42
    )
    reduced_embeddings = reducer.fit_transform(embeddings)
    print(f" UMAP complete! Shape: {reduced_embeddings.shape}\n")

    # 2. HDBSCAN
    print(" Running HDBSCAN clustering...")
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=8,
        min_samples=4,
        metric='euclidean',
        cluster_selection_method='eom'
    )
    cluster_labels = clusterer.fit_predict(reduced_embeddings)
    confidence_scores = clusterer.probabilities_
    confidence_scores[cluster_labels == -1] = 0.0

    df['subcategory_id'] = cluster_labels
    df['confidence_score'] = confidence_scores

    # Save to disk permanently
    np.save('tech_cluster_labels.npy', cluster_labels)
    np.save('tech_reduced_embeddings.npy', reduced_embeddings)
    np.save('tech_confidence_scores.npy', confidence_scores)
    print(" Results saved to disk permanently!")

    n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
    n_noise = list(cluster_labels).count(-1)

    print(f" HDBSCAN complete!")
    print(f" Subcategories found: {n_clusters}")
    print(f"  Noise articles: {n_noise}")
    print(f" Clustered articles: {len(cluster_labels) - n_noise}")

# Cluster distribution
print(f"\n Articles per subcategory:")
print(df['subcategory_id'].value_counts().sort_index())

 Loading saved embeddings...
 Embeddings shape: (401, 1024)
 Loading saved cluster results...
 Subcategories found: 18
  Noise articles: 36
 Clustered articles: 365

 Articles per subcategory:
subcategory_id
-1     36
 0     10
 1     21
 2     27
 3     17
 4     17
 5      9
 6     15
 7     43
 8     23
 9     18
 10    24
 11    13
 12    19
 13    18
 14    30
 15    40
 16    13
 17     8
Name: count, dtype: int64


## Phase 5: Thematic Labeling (c-TF-IDF)
In this phase we extract the most unique and discriminative keywords for each cluster using c-TF-IDF. This compares word frequencies across clusters to surface words that are characteristic of each subcategory rather than just globally common words.

In [5]:
# ============================================================
# PHASE 5: THEMATIC LABELING (c-TF-IDF)
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import numpy as np

# 1. Separate clustered articles (exclude noise label -1)
clustered_df = df[df['subcategory_id'] != -1].copy()

# 2. Group all articles in each cluster into one big document
cluster_documents = clustered_df.groupby('subcategory_id')['cleaned_text'].apply(
    lambda texts: ' '.join(texts)
).reset_index()

# 3. c-TF-IDF — Compare word frequencies ACROSS clusters
# This finds words that are unique to each cluster (not just common everywhere)
vectorizer = TfidfVectorizer(
    max_features=10000,
    stop_words='english',
    ngram_range=(1, 2),    # Include bigrams like "artificial intelligence", "cyber security"
    min_df=1
)

tfidf_matrix = vectorizer.fit_transform(cluster_documents['cleaned_text'])
feature_names = vectorizer.get_feature_names_out()

# 4. Extract top keywords per cluster
def get_top_keywords(tfidf_matrix, feature_names, n=10):
    top_keywords = []
    for i in range(tfidf_matrix.shape[0]):
        row = tfidf_matrix[i].toarray()[0]
        top_indices = row.argsort()[-n:][::-1]
        keywords = [feature_names[idx] for idx in top_indices]
        top_keywords.append(keywords)
    return top_keywords

top_keywords = get_top_keywords(tfidf_matrix, feature_names, n=10)

# 5. Display keywords per cluster
print(" Top keywords per subcategory:\n")
for i, row in cluster_documents.iterrows():
    cluster_id = row['subcategory_id']
    keywords = top_keywords[i]
    print(f"Cluster {cluster_id:2d} | {', '.join(keywords)}")

# 6. Add top 3 keywords as semantic label to DataFrame
cluster_documents['semantic_label'] = [
    ' | '.join(kws[:3]) for kws in top_keywords
]

# 7. Map labels back to main DataFrame
label_map = dict(zip(cluster_documents['subcategory_id'], cluster_documents['semantic_label']))
df['semantic_label'] = df['subcategory_id'].map(label_map)
df['semantic_label'] = df['semantic_label'].fillna('Noise')

print(f"\n Semantic labels assigned!")
print(f"\n Cluster ID → Semantic Label:")
for cluster_id, label in sorted(label_map.items()):
    print(f"  Cluster {cluster_id:2d} → {label}")

 Top keywords per subcategory:

Cluster  0 | patents, patent, directive, parliament, eu, law, european parliament, inventions, software, patenting
Cluster  1 | peer, p2p, said, bittorrent, file sharing, file, peer peer, music, sharing, networks
Cluster  2 | search, blogs, yahoo, google, blog, said, people, ask jeeves, jeeves, web
Cluster  3 | ds, nintendo, sony, psp, handheld, games, gaming, console, san andreas, andreas
Cluster  4 | robot, people, robots, voice, wong, said, brain, asimo, mr wong, opera
Cluster  5 | apple, lawsuit, eff, printer, journalists, cartridges, reporters, ink, information, court
Cluster  6 | spyware, security, microsoft, virus, software, windows, programs, anti, users, viruses
Cluster  7 | spam, virus, said, mail, domain, site, spammers, lycos, infected, malicious
Cluster  8 | security, said, attacks, data, net, people, rfid, tags, information, mr
Cluster  9 | ces, people, said, digital, technology, consumer, consumer electronics, electronics, technologies, de

## Phase 5b: Article-Level Keyword Extraction (KeyBERT)
In this phase we extract article-level keywords using KeyBERT for the **tech category** (401 articles, 17 subcategories, 22 noise articles). Unlike c-TF-IDF which operates at the cluster level, KeyBERT extracts keywords that best represent each individual article — providing a richer and more explainable output.

In [6]:
# ============================================================
# PHASE 5B: KEYBERT + CLUSTER VERIFICATION (TECH CATEGORY)
# ============================================================
from keybert import KeyBERT
import warnings
warnings.filterwarnings('ignore')

# 1. KeyBERT keyword extraction
print(" Loading KeyBERT model...")
kw_model = KeyBERT()
print(" KeyBERT model loaded!\n")

print(" Extracting keywords for each tech article...")
def extract_keywords(text, n=5):
    try:
        keywords = kw_model.extract_keywords(
            str(text),
            keyphrase_ngram_range=(1, 2),
            stop_words='english',
            top_n=n
        )
        return ', '.join([kw[0] for kw in keywords])
    except:
        return ''

df['article_keywords'] = df['cleaned_text'].apply(extract_keywords)
print(f" Keywords extracted for all {len(df)} tech articles!\n")

# 2. Cluster verification — read before naming
print("=" * 70)
print("   TECH CLUSTER VERIFICATION — READ BEFORE NAMING")
print("   401 articles | 17 subcategories | 22 noise articles")
print("=" * 70)

#  FLAGGED CLUSTERS — review carefully before naming:
# Cluster  2 → 87 articles (unusually large — possible split needed)
# Cluster  4 → 10 articles — mixed topics (ink/apple/eff/elections) — possible Noise
# Cluster  6 → 10 articles — mixed topics (hip hop/spanish/podcasting) — possible Noise
# Cluster 10 → 18 articles — verify separation from Cluster 11 (both gaming-related)
# Cluster 11 → 23 articles — verify separation from Cluster 10 (both gaming-related)

FLAGGED_CLUSTERS = {
    2:  "  FLAGGED — 87 articles, unusually large. Check if topic is genuinely unified or needs splitting.",
    4:  "  FLAGGED — mixed topics (printer, Apple, EFF, elections). Likely Noise.",
    6:  "  FLAGGED — mixed topics (hip hop, Spanish, podcasting). Likely Noise.",
    10: "  FLAGGED — verify separation from Cluster 11 (both gaming-related).",
    11: "  FLAGGED — verify separation from Cluster 10 (both gaming-related).",
}

for cluster_id in sorted(df[df['subcategory_id'] != -1]['subcategory_id'].unique()):
    cluster_articles = df[df['subcategory_id'] == cluster_id]

    print(f"\n{'='*70}")
    print(f"CLUSTER {cluster_id} — {len(cluster_articles)} articles")

    if cluster_id in FLAGGED_CLUSTERS:
        print(f"{FLAGGED_CLUSTERS[cluster_id]}")

    print(f"{'='*70}")

    # Sample articles — focus on cleaned_text
    samples = cluster_articles.sample(min(3, len(cluster_articles)), random_state=42)
    for i, (_, row) in enumerate(samples.iterrows(), 1):
        print(f"\n Title {i}: {row['title']}")
        print(f" Keywords: {row['article_keywords']}")
        snippet = ' '.join(str(row['cleaned_text']).split()[:80])
        print(f" Text: {snippet}...")

    print(f"\n  WHAT LABEL SHOULD THIS CLUSTER GET?")
    print(f"{'='*70}")

 Loading KeyBERT model...
 KeyBERT model loaded!

 Extracting keywords for each tech article...
 Keywords extracted for all 401 tech articles!

   TECH CLUSTER VERIFICATION — READ BEFORE NAMING
   401 articles | 17 subcategories | 22 noise articles

CLUSTER 0 — 10 articles

 Title 1: 'No re-draft' for EU patent law
 Keywords: eu legislation, proposed european, directive eu, european commission, patents drafted
 Text: A proposed European law on software patents will not be re-drafted by the European Commission (EC) despite requests by MEPs. The law is proving controversial and has been in limbo for a year. Some major tech firms say it is needed to protect inventions, while others fear it will hurt smaller tech firms. The EC says the Council of Ministers will adopt a draft version that was agreed upon last May but said it would review "all aspects of the...

 Title 2: Open source leaders slam patents
 Keywords: software patents, patents comes, patent lawsuits, linux freely, saying micros

## Phase 5c: Post KeyBERT Flagged Cluster Review (Tech Category)

In this phase we perform a deeper manual review of clusters that 
remained ambiguous after KeyBERT keyword extraction. Up to 6 sample 
articles are displayed per cluster including title, keywords and an 
80 word text snippet. The review covers 5 flagged clusters with mixed 
or unclear content. This ensures that final subcategory labels 
assigned in Phase 6 are based on a thorough understanding of each 
cluster's actual content rather than keywords alone.

In [7]:
# ============================================================
# EXTRA SAMPLES — FLAGGED CLUSTERS (7, 9, 10, 13, 14)
# ============================================================

FLAGGED = {
    7:  "mixed (handheld Kenya, supercomputing, recycling)",
    9:  "mixed (digital hoarding, gigapixel, 3G phones)",
    10: "Xbox hardware vs game reviews — verify split with Cluster 11",
    13: "mixed (tech/arts, Sony award, CES gadgets)",
    14: "assistive tech — duplicate article detected",
}

for cluster_id in FLAGGED:
    cluster_articles = df[df['subcategory_id'] == cluster_id]
    
    print(f"\n{'='*70}")
    print(f"CLUSTER {cluster_id} — {len(cluster_articles)} articles")
    print(f"  {FLAGGED[cluster_id]}")
    print(f"{'='*70}")
    
    # Print more samples this time — up to 6
    samples = cluster_articles.sample(min(6, len(cluster_articles)), random_state=99)
    for i, (_, row) in enumerate(samples.iterrows(), 1):
        print(f"\n Title {i}: {row['title']}")
        print(f" Keywords: {row['article_keywords']}")
        snippet = ' '.join(str(row['cleaned_text']).split()[:80])
        print(f" Text: {snippet}...")
    
    print(f"\n  DECISION?")
    print(f"{'='*70}")


CLUSTER 7 — 43 articles
  mixed (handheld Kenya, supercomputing, recycling)

 Title 1: Bad e-mail habits sustains spam
 Keywords: buy spam, spam mails, mails spammers, behaviour spammers, spam industry
 Text: The 'bad behaviour' of e-mail users is helping to sustain the spam industry, a new study has found. According to a survey conducted by security firm Mirapoint and market research company the Radicati Group, nearly a third of e-mail users have clicked on links in spam messages. One in ten users have bought products advertised in junk mail. Clicking on a link in a spam message can expose people to viruses and alert spammers to live e-mail accounts. The fact...

 Title 2: Fast moving phone bugs appear
 Keywords: phone viruses, phones infected, phone virus, cabir virus, symbian phones
 Text: Security firms are warning about several mobile phone viruses that can spread much faster than similar bugs. The new strains of the Cabir mobile phone virus use short-range radio technology to le

## Phase 6: Manual Label Assignment & Final Output (Tech Category)

In this phase we map each cluster ID to a human readable subcategory 
label using AI generated names. Confidence scores are applied to each 
article based on its cluster assignment. Articles assigned to noise 
clusters receive a confidence score of 0.0. The final output dataframe 
is built with all relevant columns and saved to CSV for use in 
downstream phases.

In [8]:
# ============================================================
## Phase 6: Manual Label Assignment & Final Output (TECH CATEGORY)
# ============================================================

# 1. AI-generated subcategory names for 17 confirmed clusters
ai_label_map = {
    0:  "Software Patents",
    1:  "Internet Search Engines",
    2:  "Cybersecurity",
    3:  "File Sharing",
    4:  "Assistive Technology",
    5:  "Consumer Electronics",
    6:  "Cybersecurity",
    7:  "Emerging Technology",
    8:  "Handheld Gaming",
    9:  "Mobile Technology",
    10: "Computing Hardware",
    11: "Video Games Industry",
    12: "High Definition Video",
    13: "Digital Television",
    14: "Broadband Internet",
    15: "Mobile Technology",
    16: "Consumer Electronics",
    -1: "Noise"
}

# 2. Apply confidence scores
df['confidence_score'] = confidence_scores
df.loc[df['subcategory_id'] == -1, 'confidence_score'] = 0.0

# 3. Apply AI labels
df['semantic_label'] = df['subcategory_id'].map(ai_label_map)

# 4. Build final output DataFrame cleanly
final_df = df[[
    'file_name',
    'article_id',
    'title',
    'cleaned_text',
    'subcategory_id',
    'semantic_label',
    'confidence_score',
    'article_keywords'
]].copy().reset_index(drop=True)

# 5. Save to CSV
final_df.to_csv('tech_subcategories_output.csv', index=False)

# 6. Summary
print("=" * 55)
print("       TECH CLASSIFIER — FINAL RESULTS")
print("=" * 55)
print(f" Total Articles:        {len(final_df)}")
print(f" Subcategories Found:   {final_df[final_df['semantic_label'] != 'Noise']['subcategory_id'].nunique()}")
print(f" Clustered Articles:    {len(final_df[final_df['semantic_label'] != 'Noise'])}")
print(f"  Noise Articles:        {len(final_df[final_df['semantic_label'] == 'Noise'])}")
print(f" Avg Confidence:        {final_df[final_df['semantic_label'] != 'Noise']['confidence_score'].mean():.2f}")
print("=" * 55)
print(f"\n Label Distribution:")
print(final_df['semantic_label'].value_counts())
print(f"\n Saved to tech_subcategories_output.csv")

       TECH CLASSIFIER — FINAL RESULTS
 Total Articles:        401
 Subcategories Found:   18
 Clustered Articles:    365
  Noise Articles:        36
 Avg Confidence:        0.87

 Label Distribution:
semantic_label
Mobile Technology          58
Emerging Technology        43
Cybersecurity              42
Noise                      36
Broadband Internet         30
Computing Hardware         24
Handheld Gaming            23
Consumer Electronics       22
Internet Search Engines    21
High Definition Video      19
Digital Television         18
File Sharing               17
Assistive Technology       17
Video Games Industry       13
Software Patents           10
Name: count, dtype: int64

 Saved to tech_subcategories_output.csv


## Phase 7: Manual Noise Correction (Tech Category)

In this phase we manually assign all noise articles that HDBSCAN could 
not confidently classify into a cluster. Each article was reviewed 
individually by reading the full article text and keywords before 
assignment. Corrections were applied using two methods: title based 
matching for articles identified during manual review, and article ID 
based matching for remaining noise articles identified by diagnostic 
analysis.

In [9]:
# ============================================================
# PHASE 7: MANUAL NOISE CORRECTION (TECH CATEGORY)
# ============================================================

import pandas as pd

df = pd.read_csv('tech_subcategories_output.csv')

manual_assignments = {

    # Video Games Industry
    'Xbox power cable \'fire fear\'':           'Video Games Industry',
    'Game firm holds \'cast\' auditions':        'Video Games Industry',
    'Xbox 2 may be unveiled in summer':          'Video Games Industry',
    'Games firms \'face tough future\'':         'Video Games Industry',
    'The Force is strong in Battlefront':        'Video Games Industry',
    'GTA sequel is criminally good':             'Video Games Industry',
    'Halo fans\' hope for sequel':               'Video Games Industry',
    'No half measures with Half-Life 2':         'Video Games Industry',
    'Halo 2 sells five million copies':          'Video Games Industry',
    '2D Metal Slug offers retro fun':            'Video Games Industry',
    'Gritty return for Prince of Persia':        'Video Games Industry',
    'The gaming world in 2005':                  'Video Games Industry',
    'Bond game fails to shake or stir':          'Video Games Industry',
    'Blinx sequel purrs nicely':                 'Video Games Industry',
    'Big war games battle it out':               'Video Games Industry',
    'What\'s next for next-gen consoles?':       'Video Games Industry',
    'Game makers get Xbox 2 sneak peek':         'Video Games Industry',
    'New consoles promise big problems':         'Video Games Industry',
    'Gangsters dominate gaming chart':           'Video Games Industry',
    'Argonaut founder rebuilds empire':          'Video Games Industry',

    # Consumer Electronics
    'Apple iPod family expands market':          'Consumer Electronics',
    'Millions buy MP3 players in US':            'Consumer Electronics',
    'iTunes user sues Apple over iPod':          'Consumer Electronics',
    'ITunes user sues Apple over iPod':          'Consumer Electronics',
    'Consumers \'snub portable video\'':         'Consumer Electronics',
    'Speak easy plan for media players':         'Consumer Electronics',

    # Mobile Technology
    'Moving mobile improves golf swing':         'Mobile Technology',
    'Mobile audio enters new dimension':         'Mobile Technology',
    'Mobile music challenges \'iPod age\'':      'Mobile Technology',
    'Tough rules for ringtone sellers':          'Mobile Technology',
    'Mobile gaming takes off in India':          'Mobile Technology',
    'Mobile TV tipped as one to watch':          'Mobile Technology',

    # Broadband Internet
    'Podcasts mark rise of DIY radio':           'Broadband Internet',
    'British Library gets wireless net':         'Broadband Internet',
    'Halo 2 heralds traffic explosion':          'Broadband Internet',
    'Souped-up wi-fi is on the horizon':         'Broadband Internet',
    'Ultra fast wi-fi nears completion':         'Broadband Internet',
    '\'Podcasters\' look to net money':          'Broadband Internet',

    # Emerging Technology
    'Latest Opera browser gets vocal':           'Emerging Technology',
    'Humanoid robot learns how to run':          'Emerging Technology',
    'Video phone help for deaf people':          'Emerging Technology',
    'Robots learn \'robotiquette\' rules':       'Emerging Technology',
    '\'Brainwave\' cap controls computer':       'Emerging Technology',
    'Robotic pods take on car design':           'Emerging Technology',
    'Tech helps disabled speed demons':          'Emerging Technology',
    'Anti-tremor mouse stops PC shakes':         'Emerging Technology',
    'Hitachi unveils \'fastest robot\'':         'Emerging Technology',
    'Blind student \'hears in colour\'':         'Emerging Technology',
    'Putting a face to \'Big Brother\'':         'Emerging Technology',
    'When technology gets personal':             'Emerging Technology',
    'When invention turns to innovation':        'Emerging Technology',
    'How to make a greener computer':            'Emerging Technology',

    # Assistive Technology
    'Video phone help for deaf people':          'Assistive Technology',

    # File Sharing
    'Apple sues \'Tiger\' file sharers':         'File Sharing',
    'Format wars could \'confuse users\'':       'File Sharing',

    # Cybersecurity
    'Hacker threat to Apple\'s iTunes':          'Cybersecurity',
    'Call for action on internet scam':          'Cybersecurity',
    'BT program to beat dialler scams':          'Cybersecurity',

    # Software Patents
    'Apple sues to stop product leaks':          'Software Patents',

    # High Definition Video
    'DVD copy protection strengthened':          'High Definition Video',

    # Digital Television
    'Confusion over high-definition TV':         'Digital Television',

    # Internet Regulation
    'China net cafe culture crackdown':          'Internet Regulation',
    'China \'to overtake US net use\'':          'Internet Regulation',
    'China \'blocks Google news site\'':         'Internet Regulation',
    'China \'ripe\' for media explosion':        'Internet Regulation',

    # Tech Legal Disputes
    'US woman sues over cartridges':             'Tech Legal Disputes',
    'Apple attacked over sources row':           'Tech Legal Disputes',
    'Apple makes blogs reveal sources':          'Tech Legal Disputes',
    'US woman sues over ink cartridges':         'Tech Legal Disputes',

    # Technology in Elections
    'Ink helps drive democracy in Asia':         'Technology in Elections',

    # Internet Search Engines
    'Firefox browser takes on Microsoft':        'Internet Search Engines',
    'New browser wins over net surfers':         'Internet Search Engines',

    # Sports Technology
    'Piero gives rugby perspective':             'Sports Technology',

    # Digital Media Awards
    'BBC leads interactive Bafta wins':          'Digital Media Awards',

    # Digital Media Culture
    'Web radio takes Spanish rap global':        'Digital Media Culture',
    'Poles play with GameBoy \'blip-pop\'':      'Digital Media Culture',

    # Internet Applications
    'Remote control rifle range debuts':         'Internet Applications',

    # Online Commerce
    'Man auctions ad space on forehead':         'Online Commerce',

    # Engineering Technology
    'Fast lifts rise into record books':         'Engineering Technology',
}

# Apply title based assignments
print("Applying title based noise corrections...")
title_reassigned = 0
for title, label in manual_assignments.items():
    mask = df['title'] == title
    df.loc[mask, 'semantic_label'] = label
    count = mask.sum()
    if count > 0:
        title_reassigned += count
        print(f"   '{title[:50]}' -> {label}")

# Apply article ID based assignments
id_assignments = {

    # Cybersecurity
    3:   'Cybersecurity',
    7:   'Cybersecurity',
    20:  'Cybersecurity',
    36:  'Cybersecurity',
    60:  'Cybersecurity',
    83:  'Cybersecurity',
    96:  'Cybersecurity',
    122: 'Cybersecurity',
    177: 'Cybersecurity',
    188: 'Cybersecurity',
    194: 'Cybersecurity',
    250: 'Cybersecurity',
    292: 'Cybersecurity',
    308: 'Cybersecurity',
    358: 'Cybersecurity',
    377: 'Cybersecurity',
    397: 'Cybersecurity',

    # Broadband Internet
    21:  'Broadband Internet',
    42:  'Broadband Internet',
    58:  'Broadband Internet',
    107: 'Broadband Internet',
    129: 'Broadband Internet',
    136: 'Broadband Internet',
    146: 'Broadband Internet',
    150: 'Broadband Internet',
    158: 'Broadband Internet',
    175: 'Broadband Internet',
    184: 'Broadband Internet',
    205: 'Broadband Internet',
    211: 'Broadband Internet',
    226: 'Broadband Internet',
    254: 'Broadband Internet',
    262: 'Broadband Internet',
    264: 'Broadband Internet',
    275: 'Broadband Internet',
    328: 'Broadband Internet',
    339: 'Broadband Internet',
    343: 'Broadband Internet',
    350: 'Broadband Internet',
    352: 'Broadband Internet',
    392: 'Broadband Internet',

    # Emerging Technology
    4:   'Emerging Technology',
    6:   'Emerging Technology',
    19:  'Emerging Technology',
    23:  'Emerging Technology',
    33:  'Emerging Technology',
    63:  'Emerging Technology',
    66:  'Emerging Technology',
    137: 'Emerging Technology',
    161: 'Emerging Technology',
    191: 'Emerging Technology',
    199: 'Emerging Technology',
    200: 'Emerging Technology',
    223: 'Emerging Technology',
    242: 'Emerging Technology',
    307: 'Emerging Technology',
    329: 'Emerging Technology',
    331: 'Emerging Technology',
    367: 'Emerging Technology',
    380: 'Emerging Technology',
    381: 'Emerging Technology',

    # Internet Search Engines
    138: 'Internet Search Engines',
    196: 'Internet Search Engines',
    261: 'Internet Search Engines',

    # File Sharing
    285: 'File Sharing',
    304: 'File Sharing',

    # Mobile Technology
    54:  'Mobile Technology',
    67:  'Mobile Technology',
    192: 'Mobile Technology',
    378: 'Mobile Technology',
    382: 'Mobile Technology',
    383: 'Mobile Technology',

    # Digital Television
    349: 'Digital Television',
    361: 'Digital Television',
    365: 'Digital Television',
    366: 'Digital Television',
    394: 'Digital Television',

    # Video Games Industry
    15:  'Video Games Industry',
    24:  'Video Games Industry',
    52:  'Video Games Industry',
    82:  'Video Games Industry',
    119: 'Video Games Industry',
    203: 'Video Games Industry',
    309: 'Video Games Industry',
    348: 'Video Games Industry',
    396: 'Video Games Industry',

    # Consumer Electronics
    12:  'Consumer Electronics',
    18:  'Consumer Electronics',
    22:  'Consumer Electronics',
    141: 'Consumer Electronics',
    173: 'Consumer Electronics',
    190: 'Consumer Electronics',
    319: 'Consumer Electronics',
    320: 'Consumer Electronics',
    332: 'Consumer Electronics',

    # Internet Regulation
    2:   'Internet Regulation',
    74:  'Internet Regulation',
    87:  'Internet Regulation',
    186: 'Internet Regulation',
    384: 'Internet Regulation',

    # Technology in Elections
    1:   'Technology in Elections',

    # Assistive Technology
    251: 'Assistive Technology',

    # Digital Media Awards
    46:  'Digital Media Awards',

    # Internet Applications
    202: 'Internet Applications',

    # Online Commerce
    209: 'Online Commerce',
}

print("\nApplying article ID based noise corrections...")
id_reassigned = 0
for article_id, label in id_assignments.items():
    mask = df['article_id'] == article_id
    df.loc[mask, 'semantic_label'] = label
    count = mask.sum()
    if count > 0:
        id_reassigned += count

# Verify
noise_remaining = len(df[df['semantic_label'] == 'Noise'])
print(f"\nTitle based assignments:    {title_reassigned}")
print(f"Article ID assignments:     {id_reassigned}")
print(f"Noise remaining:            {noise_remaining}")
print(f"Total articles:             {len(df)}")

print(f"\nFinal Label Distribution:")
print(df['semantic_label'].value_counts())

# Save back
df.to_csv('tech_subcategories_output.csv', index=False)
print(f"\nTech noise correction complete!")

Applying title based noise corrections...
   'Xbox power cable 'fire fear'' -> Video Games Industry
   'Game firm holds 'cast' auditions' -> Video Games Industry
   'Xbox 2 may be unveiled in summer' -> Video Games Industry
   'Games firms 'face tough future'' -> Video Games Industry
   'The Force is strong in Battlefront' -> Video Games Industry
   'GTA sequel is criminally good' -> Video Games Industry
   'Halo fans' hope for sequel' -> Video Games Industry
   'No half measures with Half-Life 2' -> Video Games Industry
   'Halo 2 sells five million copies' -> Video Games Industry
   '2D Metal Slug offers retro fun' -> Video Games Industry
   'Gritty return for Prince of Persia' -> Video Games Industry
   'The gaming world in 2005' -> Video Games Industry
   'Bond game fails to shake or stir' -> Video Games Industry
   'Blinx sequel purrs nicely' -> Video Games Industry
   'Big war games battle it out' -> Video Games Industry
   'What's next for next-gen consoles?' -> Video Games Indu

## Phase 8: Named Entity Recognition with April Events Extraction
In this phase we address the project's **Desired** requirement from the project brief:

> *"Identify documents and extract the named entities for media personalities, clearly identifying their jobs (e.g. Politicians, TV/Film Personalities, Musicians)"*

> *"Extract summaries of anything that took place or is/was scheduled to take place in April."*

We use spaCy to extract named entities from each article's combined `title` and `cleaned_text` for the **tech category** (401 articles). First we verify spaCy is installed and the English language model is available.

## Phase 8A: Named Entity Recognition — Persons, Organisations, Locations (Tech Category)
In this phase we extract named entities from each article's combined `title` and `cleaned_text` using spaCy's `en_core_web_lg` model across all 401 tech articles. Three entity types are extracted:

- **Named Persons** — validated through a false positive filter that removes game titles, movie titles, software names, month names, acronyms, and single-word misclassifications, ensuring only real two-word-minimum person names are retained
- **Named Organisations** — companies, institutions, regulatory bodies, and other organisations mentioned
- **Named Locations** — countries, cities, and regions identified as GPE or LOC entities

Each valid person is also classified by role using a context window of 80 characters surrounding the entity, matching against keyword lists for roles including Politician, Business Executive, Central Banker, Expert/Analyst, TV/Film Personality, Musician, Sports Personality, Journalist, and Technology Professional. Results: **354** articles with named persons, **400** with organisations, **355** with locations.

In [25]:
# ============================================================
# PHASE 9A: NER — PERSONS, ORGS, LOCATIONS (TECH CATEGORY)
# ============================================================
import spacy
import pandas as pd

nlp = spacy.load('en_core_web_lg')
df = pd.read_csv('tech_subcategories_output.csv')
print(f" Loaded {len(df)} tech articles")

# ============================================================
# FALSE POSITIVE FILTER
# ============================================================
FALSE_POSITIVES = {
    # Movie/TV/Game titles
    'alexander', 'catwoman', 'rings', 'halo', 'titanic',
    'batman', 'superman', 'spiderman', 'matrix', 'avatar',
    'shrek', 'gladiator', 'braveheart', 'terminator',
    'blinx', 'xenon', 'painkiller', 'battlefront',
    'goldeneye', 'goldfinger', 'napster', 'kazaa',
    # Game/fictional characters
    'darth vader', 'luke skywalker', 'jedi knight', 'peter parker',
    'james bond', 'auric goldfinger', 'xenia onatopp', 'metal slug',
    'metal slug 3', 'harry potter', 'finding nemo', 'doc ock',
    'super mario', 'daxter and ratchet and clank', 'doom iii',
    'anakin skywalker', 'bob cratchit', 'arthur dent',
    # Product/brand names tagged as persons
    'nokia 6600', 'qrio robot', 'mac mini', 'spider-man 2',
    'bt openzone', 'virgin megastores', 'virgin radio',
    'virgin trains later in the year', 'phoenix torrent',
    'phoenix torrents', 'pal and chum', 'free mojtaba',
    "arash day'", 'ray dvd', 'my ask jeeves',
    'los caballeros de plan g', 'drew show',
    'de waarschuwingsdienst', 'nat west',
    # Possessives that slip through
    "ms simonetti's", "adam curry's", "george w bush's",
    "ms fiorina's", "george lucas'", "bill gates'",
    "mr senanayake's", "prof reddy's", "james martin's",
    "james bond's", "andrei tarkovsky's", "mr van gogh's",
    "ms stefani", "ron bloom's",
    # Honorific-only single names
    'mr negroponte', 'mr stone', 'ms perry', 'dr paniccia',
    'mr stanley', 'ms sanders', 'mr sridharan', 'mr clark',
    'dr nobles', 'mr ash', 'mr pearson', 'ms hanson',
    'ms rellie', 'mr makower', 'ms milanesi', 'mr hosford',
    'mr short', 'mr houlihan', 'ms musselman', 'mr chandratillake',
    'mr bud', 'mr jain', 'mr cox', 'mr fogg', 'mr dean',
    'mr armes', 'mr allard', 'mr morgan', 'mr donofrio',
    'mr seagrave', 'mr morgenstern', 'mr pouwelse',
    'dr frasca', 'dr bowden', 'dr bjorn', 'dr avineri',
    'dr gillingwater', 'ms harker', 'ms gambino',
    'ms finger', 'ms taylor', 'ms everett', 'ms musselman',
    'mr king', 'mr simpson', 'mr palmer', 'mr gupta',
    'mr mehta', 'mr macklin', 'mr trainor', 'mr underwood',
    'mr warner', 'mr walsh', 'dr carpenter', 'dr seuss',
    'mr thiemann', 'mr hogan', 'mr doctorow', 'mr davis',
    'mr oudendijk', 'mr binks', 'dr nielsen', 'prof reddy',
    'prof brewer', 'supt deats', 'mr monteith',
    # Days/Months
    'monday', 'tuesday', 'wednesday', 'thursday', 'friday',
    'saturday', 'sunday', 'january', 'february', 'march',
    'april', 'may', 'june', 'july', 'august', 'september',
    'october', 'november', 'december',
    # Common misclassified words
    'lord', 'sir', 'mr', 'mrs', 'ms', 'dr', 'prof',
    'bertelsmann', 'aol', 'sec', 'gm', 'ibm', 'bbc',
    'linux', 'windows', 'google', 'yahoo', 'mozilla',
    'opera', 'firefox', 'quicktime', 'itunes', 'ipod',
    # Literary/historical references used as metaphors
    'childe harold',  'auld lang syne', 'holy grail',
    'john doe', 'colin mcrae', 'colin mcrae rally',
    # Fictional TV characters
    'johnny carson', 'ed sullivan', 'vicky pollard',
    'kelly holmes',
}

# Exact semantic_label values from tech dataset
TECH_LABELS = {
    'cybersecurity', 'emerging technology', 'mobile technology',
    'internet regulation', 'consumer electronics', 'video games industry',
    'digital television', 'software regulation', 'search technology',
    'broadband internet', 'e-commerce', 'artificial intelligence',
    'technology policy', 'internet culture', 'hardware',
    'open source', 'digital media', 'robotics'
}

def is_valid_person(name):
    if len(name) < 2:
        return False
    if name.lower() in FALSE_POSITIVES:
        return False
    if name.isupper():
        return False
    if not any(c.isalpha() for c in name):
        return False
    tokens = name.strip().split()
    if len(tokens) < 2:
        return False
    # filter possessives
    if name.endswith("'s") or name.endswith("s'"):
        return False
    # filter honorific-only entries
    honorifics = {'mr', 'mrs', 'ms', 'dr', 'prof', 'dame', 'sir',
                  'lord', 'supt', 'rev'}
    if tokens[0].lower().rstrip('.') in honorifics and len(tokens) == 2:
        return False
    # filter single initials like "J Smith"
    if len(tokens[0]) == 1 and tokens[0].isupper():
        return False
    # filter number entries
    if tokens[-1].isdigit():
        return False
    # filter comma-separated pairs
    if ',' in name:
        return False
    # filter entries with dots like "S. Young"
    if len(tokens[0]) <= 2 and '.' in tokens[0]:
        return False
    # filter "Name of Organisation" patterns
    if ' of ' in name.lower():
        return False
    return True

# ============================================================
# KNOWN PERSON ROLE LOOKUP
# ============================================================
KNOWN_ROLES = {
    # Technology Professionals / Business Executives
    'Bill Gates':           'Business Executive',
    'Steve Jobs':           'Business Executive',
    'Carly Fiorina':        'Business Executive',
    'Jonathan Schwartz':    'Business Executive',
    'Jonathan Schwartz':    'Business Executive',
    'Robbie Bach':          'Business Executive',
    'Simon Fuller':         'Business Executive',
    'Peter Molyneux':       'Technology Professional',
    'Jef Raskin':           'Technology Professional',
    'Andy Hertzfeld':       'Technology Professional',
    'David Filo':           'Business Executive',
    'Jerry Yang':           'Business Executive',
    'Graham Cluley':        'Technology Professional',
    'Mikko Hypponen':       'Technology Professional',
    'Stephen Toulouse':     'Technology Professional',
    'Haisheng Rong':        'Technology Professional',
    'Richard Jones':        'Technology Professional',
    'Ansheng Liu':          'Technology Professional',
    'Oded Cohen':           'Technology Professional',
    'Dani Hak':             'Technology Professional',
    'Alexander Fang':       'Technology Professional',
    'Blake Irving':         'Technology Professional',
    'Ian Pearson':          'Technology Professional',
    'Bill Thompson':        'Technology Professional',
    'Cory Doctorow':        'Technology Professional',
    'Dave Winer':           'Technology Professional',
    'Jonathan Wolpaw':      'Technology Professional',
    'Dennis McFarlane':     'Technology Professional',
    'Jakob Nielsen':        'Technology Professional',
    'Doug Lombardi':        'Technology Professional',
    'Mark Sunner':          'Technology Professional',
    'Richard Bowden':       'Technology Professional',
    'Malte Pollmann':       'Technology Professional',
    'Phil Cracknell':       'Technology Professional',
    'Scott Stanzel':        'Technology Professional',
    'Todd Thiemann':        'Technology Professional',
    'David Beckham':        'Sports Personality',
    # Expert/Analysts
    'David Mikosz':         'Expert/Analyst',
    'Shauna Perry':         'Expert/Analyst',
    'Patricia Sanders':     'Expert/Analyst',
    'Andrew Thompson':      'Expert/Analyst',
    'Jeremy Philpott':      'Expert/Analyst',
    'Jill Sutherland':      'Expert/Analyst',
    'Steve Ash':            'Expert/Analyst',
    'Alastair Brydon':      'Expert/Analyst',
    'Grant Dean':           'Expert/Analyst',
    'Jemima Rellie':        'Expert/Analyst',
    'Richard Tateson':      'Expert/Analyst',
    'Nikolai Nolan':        'Expert/Analyst',
    'Berit Hanson':         'Expert/Analyst',
    'Niel Ransom':          'Expert/Analyst',
    'Martin Owen':          'Expert/Analyst',
    'David Stern':          'Expert/Analyst',
    'Allen Wilson':         'Expert/Analyst',
    'Andrew Fanara':        'Expert/Analyst',
    'Tony Macklin':         'Expert/Analyst',
    'Jemima Rellie':        'Expert/Analyst',
    'Andrew Pearson':       'Expert/Analyst',
    'Pete Simpson':         'Expert/Analyst',
    'Paul King':            'Expert/Analyst',
    'Dan Glickman':         'Expert/Analyst',
    'Peter Smith':          'Expert/Analyst',
    'Rohit Gupta':          'Expert/Analyst',
    'Dina Mehta':           'Expert/Analyst',
    'Yadav Kotamaraja':     'Expert/Analyst',
    'Peter Stollenmayer':   'Expert/Analyst',
    'Scott Christie':       'Expert/Analyst',
    'Mick Deats':           'Expert/Analyst',
    'Richard Doherty':      'Expert/Analyst',
    'Michael Heilmann':     'Expert/Analyst',
    'Gonzalo Frasca':       'Expert/Analyst',
    'Spencer Abraham':      'Expert/Analyst',
    'Paul Arling':          'Expert/Analyst',
    'Einar Bjorgo':         'Expert/Analyst',
    'Stefan Voigt':         'Expert/Analyst',
    'Herbert Hansen':       'Expert/Analyst',
    'Stephen Candillon':    'Expert/Analyst',
    'Michael Bjorn':        'Expert/Analyst',
    'Henri Crohas':         'Expert/Analyst',
    'Mike Coleman':         'Expert/Analyst',
    'David Gillingwater':   'Expert/Analyst',
    'Erel Avineri':         'Expert/Analyst',
    'Nigel Marson':         'Expert/Analyst',
    'Rachel Harker':        'Expert/Analyst',
    'Kirsten Pfeifer':      'Expert/Analyst',
    'John Wilkin':          'Expert/Analyst',
    'Andy Quested':         'Expert/Analyst',
    'Nick Ross':            'Expert/Analyst',
    'Hideo Kojima':         'Expert/Analyst',
    'Robert Currington':    'Expert/Analyst',
    'Eva Lichtenberg':      'Expert/Analyst',
    'Yonca Brunini':        'Expert/Analyst',
    'Nick Hazel':           'Expert/Analyst',
    'Gabriele Dorries':     'Expert/Analyst',
    'Rich Templeton':       'Expert/Analyst',
    'Karen Walsh':          'Expert/Analyst',
    'James Kleinberg':      'Expert/Analyst',
    'Kurt Opsahl':          'Expert/Analyst',
    'Rudolf Fischer':       'Expert/Analyst',
    'Mike Trainor':         'Expert/Analyst',
    'David Souter':         'Expert/Analyst',
    'Olivier Gerolami':     'Expert/Analyst',
    'David Theriault':      'Expert/Analyst',
    'Matthew Herren':       'Expert/Analyst',
    'Maciej Sundra':        'Expert/Analyst',
    'Jeremy Flynn':         'Expert/Analyst',
    'David Noone':          'Expert/Analyst',
    'Diane Carr':           'Expert/Analyst',
    'Iggy Fanlo':           'Expert/Analyst',
    'Adam Hume':            'Expert/Analyst',
    'Tim Hanlon':           'Expert/Analyst',
    'Marc Morin':           'Expert/Analyst',
    'Shailendra Jain':      'Expert/Analyst',
    'Andrew Fisher':        'Expert/Analyst',
    'Tim Warner':           'Expert/Analyst',
    'Paul Goosens':         'Expert/Analyst',
    'Rob Dwight':           'Expert/Analyst',
    'Elizabeth France':     'Expert/Analyst',
    'Will Davies':          'Expert/Analyst',
    'Gerhard Romen':        'Expert/Analyst',
    'Belinda Henderson':    'Expert/Analyst',
    'Claudia Neulling':     'Expert/Analyst',
    'Susanne Risse':        'Expert/Analyst',
    'Maria Capella':        'Expert/Analyst',
    'Bob Strauss':          'Expert/Analyst',
    'Dave Karger':          'Expert/Analyst',
    'Paul Dergarabedian':   'Expert/Analyst',
    'Ravi Purushotma':      'Expert/Analyst',
    'Max Lyons':            'Expert/Analyst',
    'John Underwood':       'Expert/Analyst',
    'John Craig':           'Expert/Analyst',
    'Aaron Foster':         'Expert/Analyst',
    'David Treacy':         'Expert/Analyst',
    'Roy Tansley':          'Expert/Analyst',
    'Dave Hawkins':         'Expert/Analyst',
    'Joy Rainey':           'Expert/Analyst',
    'Travis Anderson':      'Expert/Analyst',
    'Charles MacIntyre':    'Expert/Analyst',
    'Stuart Stanton-Davies':'Expert/Analyst',
    'James Goodman':        'Expert/Analyst',
    'Georgeta Minciu':      'Expert/Analyst',
    'Arash Sigarchi':       'Expert/Analyst',
    'Mojtaba Saminejad':    'Expert/Analyst',
    'Motjaba Saminejad':    'Expert/Analyst',
    'Ellen Simonetti':      'Expert/Analyst',
    'Jason O-Grady':        'Expert/Analyst',
    'David Rubin':          'Expert/Analyst',
    'Taran Rampersad':      'Expert/Analyst',
    'Dan Lane':             'Expert/Analyst',
    'John Sankus':          'Expert/Analyst',
    'Richard Berry':        'Expert/Analyst',
    'Kent Kartadinata':     'Expert/Analyst',
    'Christopher Tresco':   'Expert/Analyst',
    'Paul McNulty':         'Expert/Analyst',
    'Hew Raymond Griffiths':'Expert/Analyst',
    'Antony Townsden':      'Expert/Analyst',
    'Alex Bell':            'Expert/Analyst',
    'Steven Dowd':          'Expert/Analyst',
    'Dean Boyd':            'Expert/Analyst',
    'Philip Wride':         'Expert/Analyst',
    'Mike Caudwell':        'Expert/Analyst',
    'Manuel Millan':        'Expert/Analyst',
    'Rohit Gupta':          'Expert/Analyst',
    'Chris McIntyre':       'Expert/Analyst',
    'Eric Brewer':          'Technology Professional',
    'Prassana Rambathla':   'Technology Professional',
    'Paul Goosens':         'Expert/Analyst',
    'Eliot Spitzer':        'Politician',
    'Greg Abbott':          'Politician',
    'Ayatollah Ali Khamenei': 'Politician',
    'Jeremy Jaynes':        'Lawyer',
    'Jessica DeGroot':      'Lawyer',
    'Richard Rutkowski':    'Lawyer',
    'Russell McGuire':      'Lawyer',
    'Jerry Kilgore':        'Lawyer',
    'David Oblon':          'Lawyer',
    'Scott Richter':        'Lawyer',
    'Steven Richter':       'Lawyer',
    'Ryan Samuel Pitylak':  'Lawyer',
    'William Trowbridge':   'Lawyer',
    'Michael Chicoine':     'Lawyer',
    'Nicholas Lee Jacobsen':'Lawyer',
    'Peter Dobrow':         'Lawyer',
    'Anatoly Tyukanov':     'Lawyer',
}

# ============================================================
# PERSON ROLE CLASSIFIER
# ============================================================
def classify_role(person, context, semantic_label=''):
    # Check known persons first
    if person in KNOWN_ROLES:
        return KNOWN_ROLES[person]

    context = context.lower()
    semantic_label = semantic_label.lower().strip() if semantic_label else ''

    if any(w in context for w in [
        'president', 'prime minister', 'minister', 'senator', 'mp ',
        'chancellor', 'governor', 'secretary of state', 'politician',
        'parliament', 'elected', 'vote', 'constituency', 'opposition',
        'cabinet', 'treasury', 'democrat', 'republican'
    ]):
        return 'Politician'
    elif any(w in context for w in [
        'central bank', 'federal reserve', 'monetary policy',
        'interest rate', 'bank of england', 'ecb', 'imf'
    ]):
        return 'Central Banker'
    elif any(w in context for w in [
        'lawyer', 'attorney', 'legal counsel', 'defendant',
        'prosecution', 'lawsuit', 'sued', 'trial', 'verdict',
        'indicted', 'charged', 'acquitted', 'solicitor', 'barrister',
        'judge', 'court', 'convicted', 'sentence', 'prison', 'jail'
    ]):
        return 'Lawyer'
    elif any(w in context for w in [
        'actor', 'actress', 'film', 'movie', 'tv', 'television',
        'presenter', 'host', 'comedian', 'starred', 'cast', 'screen'
    ]):
        return 'TV/Film Personality'
    elif any(w in context for w in [
        'singer', 'musician', 'rapper', 'band', 'music', 'album',
        'song', 'artist', 'record', 'chart', 'tour', 'concert'
    ]):
        return 'Musician'
    elif any(w in context for w in [
        'ceo', 'chief executive', 'chairman', 'founder', 'executive',
        'managing director', 'cfo', 'coo', 'board of directors',
        'shareholder', 'merger', 'acquisition', 'listed', 'stock',
        'shares', 'profit', 'revenue', 'venture capital'
    ]):
        return 'Business Executive'
    elif any(w in context for w in [
        'developer', 'engineer', 'programmer', 'coder', 'software',
        'technical', 'architect', 'hacker', 'security researcher',
        'open source', 'algorithm', 'kernel', 'firmware', 'hardware',
        'network', 'protocol', 'encryption', 'antivirus', 'malware',
        'virus', 'spyware', 'firewall', 'blog', 'blogger', 'podcast',
        'webcam', 'broadband', 'wi-fi', 'wireless', 'router',
        'chip', 'processor', 'semiconductor', 'laser', 'robot',
        'console', 'gadget', 'device', 'mobile', 'handset'
    ]):
        return 'Technology Professional'
    elif any(w in context for w in [
        'analyst', 'economist', 'researcher', 'professor', 'scientist',
        'expert', 'academic', 'lecturer', 'scholar', 'consultant',
        'adviser', 'advisor', 'specialist', 'director of', 'think tank',
        'institute', 'forecast', 'study', 'report'
    ]):
        return 'Expert/Analyst'
    elif any(w in context for w in [
        'coach', 'manager', 'player', 'footballer', 'athlete',
        'champion', 'scored', 'match', 'tournament', 'olympic',
        'medal', 'world record', 'championship'
    ]):
        return 'Sports Personality'
    elif any(w in context for w in [
        'journalist', 'reporter', 'editor', 'correspondent', 'critic',
        'columnist', 'broadcaster', 'commentator', 'pundit', 'anchor',
        'newspaper', 'magazine', 'media', 'press', 'author', 'writer',
        'book', 'novel', 'published'
    ]):
        return 'Journalist'
    # Subcategory fallback — tech labels default to Technology Professional
    elif semantic_label in TECH_LABELS:
        return 'Technology Professional'
    else:
        return 'Other'

# ============================================================
# NER EXTRACTION FUNCTION
# ============================================================
def extract_persons_orgs_locations(title, cleaned_text, semantic_label=''):
    combined = f"{title}. {cleaned_text}"
    doc = nlp(combined[:100000])

    persons = []
    person_roles = []
    orgs = []
    locations = []

    seen_persons = set()
    seen_orgs = set()
    seen_locations = set()

    for ent in doc.ents:

        if ent.label_ == 'PERSON':
            if ent.text not in seen_persons and is_valid_person(ent.text):
                seen_persons.add(ent.text)
                start = max(0, ent.start_char - 300)
                end = min(len(combined), ent.end_char + 300)
                context = combined[start:end]
                role = classify_role(ent.text, context, semantic_label)
                persons.append(ent.text)
                person_roles.append(f"{ent.text}: {role}")

        elif ent.label_ == 'ORG':
            if ent.text not in seen_orgs:
                org_text = ent.text.strip()
                if org_text.count('(') > org_text.count(')'):
                    org_text += ')'
                if org_text.count('[') > org_text.count(']'):
                    org_text += ']'
                seen_orgs.add(org_text)
                orgs.append(org_text)

        elif ent.label_ in ['GPE', 'LOC']:
            if ent.text not in seen_locations:
                seen_locations.add(ent.text)
                locations.append(ent.text)

    return {
        'named_persons':       ' | '.join(persons),
        'named_organisations': ' | '.join(orgs),
        'named_locations':     ' | '.join(locations),
        'person_roles':        ' | '.join(person_roles),
    }

# ============================================================
# RUN NER ON ALL ARTICLES
# ============================================================
print(" Running NER (Persons, Orgs, Locations)...")

named_persons       = []
named_organisations = []
named_locations     = []
person_roles        = []

for i, row in df.iterrows():
    result = extract_persons_orgs_locations(
        str(row['title']),
        str(row['cleaned_text']),
        str(row.get('semantic_label', ''))
    )
    named_persons.append(result['named_persons'])
    named_organisations.append(result['named_organisations'])
    named_locations.append(result['named_locations'])
    person_roles.append(result['person_roles'])

    if (i + 1) % 50 == 0:
        print(f"   Processed {i + 1}/{len(df)} articles...")

df['named_persons']       = named_persons
df['named_organisations'] = named_organisations
df['named_locations']     = named_locations
df['person_roles']        = person_roles

print(f"\n Person/Org/Location NER complete!")
print(f" Articles with Persons:   {sum(1 for x in named_persons if x)}")
print(f" Articles with Orgs:      {sum(1 for x in named_organisations if x)}")
print(f" Articles with Locations: {sum(1 for x in named_locations if x)}")

# ============================================================
# AUDIT — check for remaining 'Other'
# ============================================================
other_count = df['person_roles'].str.contains('Other', na=False).sum()
print(f"\n Articles still containing 'Other': {other_count}")

other_rows = df[df['person_roles'].str.contains("Other", na=False)].copy()

def extract_others(person_roles_str):
    entries = person_roles_str.split(" | ")
    return [e.strip() for e in entries if e.strip().endswith(": Other")]

other_rows['other_persons'] = other_rows['person_roles'].apply(extract_others)

for _, row in other_rows.iterrows():
    print(f"Article {row['article_id']} | {row['title'][:60]}")
    for person in row['other_persons']:
        print(f"   --> {person}")
    print()

 Loaded 401 tech articles
 Running NER (Persons, Orgs, Locations)...
   Processed 50/401 articles...
   Processed 100/401 articles...
   Processed 150/401 articles...
   Processed 200/401 articles...
   Processed 250/401 articles...
   Processed 300/401 articles...
   Processed 350/401 articles...
   Processed 400/401 articles...

 Person/Org/Location NER complete!
 Articles with Persons:   342
 Articles with Orgs:      400
 Articles with Locations: 355

 Articles still containing 'Other': 0


## Phase 8B: April Events Extraction (Tech Category)
In this phase we extract April-related events from each article using spaCy's DATE entity recognition. For every article where a DATE entity containing "April" is detected in the combined `title` and `cleaned_text`, a 300-character contextual window surrounding the mention is captured as the event summary. This satisfies the project's Desired requirement of identifying events that took place or were scheduled to take place in April. Results: **15 out of 401** tech articles contained April events.

In [26]:
# ============================================================
# PHASE 8B: NER — APRIL EVENTS (TECH CATEGORY)
# ============================================================

def extract_april_events(title, cleaned_text):
    combined = f"{title}. {cleaned_text}"
    doc = nlp(combined[:100000])

    april_events = []

    for ent in doc.ents:
        if ent.label_ == 'DATE':
            if 'april' in ent.text.lower():
                start = max(0, ent.start_char - 100)
                end = min(len(combined), ent.end_char + 200)
                april_events.append(combined[start:end].strip())

    return ' | '.join(april_events) if april_events else ''

print(" Extracting April events...")

april_events = []
for i, row in df.iterrows():
    result = extract_april_events(str(row['title']), str(row['cleaned_text']))
    april_events.append(result)

    if (i + 1) % 50 == 0:
        print(f"   Processed {i + 1}/{len(df)} articles...")

df['april_events'] = april_events

print(f"\n April Events extraction complete!")
print(f"  Articles with April Events: {sum(1 for x in april_events if x)}")

 Extracting April events...
   Processed 50/401 articles...
   Processed 100/401 articles...
   Processed 150/401 articles...
   Processed 200/401 articles...
   Processed 250/401 articles...
   Processed 300/401 articles...
   Processed 350/401 articles...
   Processed 400/401 articles...

 April Events extraction complete!
  Articles with April Events: 15


## Phase 8C: Save Complete NER Output (Tech Category)
In this phase we consolidate all extracted NER columns — `named_persons`, `named_organisations`, `named_locations`, `person_roles`, and `april_events` — into the final dataframe and save to `tech_ner_output.csv`. A sample verification of the first article is printed to confirm all columns are correctly populated before proceeding to the next category.

In [27]:
# ============================================================
# PHASE 8C: SAVE COMPLETE NER OUTPUT (TECH CATEGORY)
# ============================================================

df['named_persons']       = named_persons
df['named_organisations'] = named_organisations
df['named_locations']     = named_locations
df['person_roles']        = person_roles
df['april_events']        = april_events

df.to_csv('tech_ner_output.csv', index=False)
print(" Saved to tech_ner_output.csv")
print(f" Columns: {df.columns.tolist()}")
print(f"\n Sample row 1:")
print(f" Persons:  {df['named_persons'].iloc[0]}")
print(f" Orgs:     {df['named_organisations'].iloc[0]}")
print(f" Location: {df['named_locations'].iloc[0]}")
print(f" Roles:    {df['person_roles'].iloc[0]}")
print(f"  April:    {df['april_events'].iloc[0]}")

print("\n" + "=" * 55)
print("        TECH NER — FINAL SUMMARY")
print("=" * 55)
print(f" Total Articles:             {len(df)}")
print(f" Articles with Persons:      {sum(1 for x in named_persons if x)}")
print(f" Articles with Orgs:         {sum(1 for x in named_organisations if x)}")
print(f" Articles with Locations:    {sum(1 for x in named_locations if x)}")
print(f"  Articles with April Events: {sum(1 for x in april_events if x)}")
print("=" * 55)

 Saved to tech_ner_output.csv
 Columns: ['file_name', 'article_id', 'title', 'cleaned_text', 'subcategory_id', 'semantic_label', 'confidence_score', 'article_keywords', 'named_persons', 'named_organisations', 'named_locations', 'person_roles', 'april_events']

 Sample row 1:
 Persons:  Askar Akaev | David Mikosz
 Orgs:     the German Embassy | the Soros Foundation | Coalition of Non-governmental Organizations | IFES
 Location: Asia | The Kyrgyz Republic | US | Ukraine | Georgia | Serbia | South Africa | Indonesia | Turkey | Afghanistan
 Roles:    Askar Akaev: Politician | David Mikosz: Expert/Analyst
  April:    

        TECH NER — FINAL SUMMARY
 Total Articles:             401
 Articles with Persons:      342
 Articles with Orgs:         400
 Articles with Locations:    355
  Articles with April Events: 15


## **** Final Project Summary — BBC NLP Project 2

This cell generates the final consolidated summary across all five 
categories: Business, Tech, Sport, Politics and Entertainment. It reads 
from each category's NER output CSV and computes the total articles, 
subcategory count, average confidence score, named entity coverage and 
April events count per category. A full subcategory distribution is 
printed for each category followed by overall project totals aggregated 
across all 2207 articles.

In [16]:
# ============================================================
# **** FINAL PROJECT SUMMARY — BBC NLP PROJECT 2
# ============================================================
import pandas as pd

categories = {
    'Business':      'business_ner_output.csv',
    'Tech':          'tech_ner_output.csv',
    'Sport':         'sport_ner_output.csv',
    'Politics':      'politics_ner_output.csv',
    'Entertainment': 'entertainment_ner_output.csv',
}

print("=" * 70)
print("         BBC NLP PROJECT 2 — FINAL CONSOLIDATED SUMMARY")
print("=" * 70)

total_articles = 0
total_subcategories = 0
total_persons = 0
total_orgs = 0
total_locations = 0
total_april = 0

for category, file in categories.items():
    df = pd.read_csv(file)

    articles = len(df)
    subcategories = df[df['semantic_label'] != 'Noise']['semantic_label'].nunique()
    persons = (df['named_persons'].notna() & (df['named_persons'] != '')).sum()
    orgs = (df['named_organisations'].notna() & (df['named_organisations'] != '')).sum()
    locations = (df['named_locations'].notna() & (df['named_locations'] != '')).sum()
    april = (df['april_events'].notna() & (df['april_events'] != '')).sum()
    avg_conf = df[df['semantic_label'] != 'Noise']['confidence_score'].mean()

    total_articles += articles
    total_subcategories += subcategories
    total_persons += persons
    total_orgs += orgs
    total_locations += locations
    total_april += april

    print(f"\n{'='*70}")
    print(f"  {category.upper()} CATEGORY")
    print(f"{'='*70}")
    print(f"   Total Articles:          {articles}")
    print(f"   Subcategories:           {subcategories}")
    print(f"   Avg Confidence:          {avg_conf:.2f}")
    print(f"   Articles with Persons:   {persons}")
    print(f"   Articles with Orgs:      {orgs}")
    print(f"   Articles with Locations: {locations}")
    print(f"   April Events:            {april}")
    print(f"\n   Subcategory Distribution:")
    dist = df['semantic_label'].value_counts()
    for label, count in dist.items():
        print(f"     {label:<35} {count}")

print(f"\n{'='*70}")
print(f"         OVERALL PROJECT TOTALS")
print(f"{'='*70}")
print(f"   Total Articles:            {total_articles}")
print(f"   Total Subcategories:       {total_subcategories}")
print(f"   Total Articles w/ Persons: {total_persons}")
print(f"   Total Articles w/ Orgs:    {total_orgs}")
print(f"   Total Articles w/ Locs:    {total_locations}")
print(f"   Total April Events:        {total_april}")
print(f"{'='*70}")
print(f"\nBBC NLP PROJECT 2 COMPLETE!")

         BBC NLP PROJECT 2 — FINAL CONSOLIDATED SUMMARY

  BUSINESS CATEGORY
   Total Articles:          510
   Subcategories:           33
   Avg Confidence:          0.74
   Articles with Persons:   433
   Articles with Orgs:      503
   Articles with Locations: 488
   April Events:            29

   Subcategory Distribution:
     Macroeconomic Indicators            82
     Corporate Governance                55
     Crude Oil Markets                   44
     Telecommunications                  42
     Automotive Industry                 31
     Financial Markets                   23
     Foreign Exchange                    20
     Aviation Industry                   20
     Macroeconomics                      19
     Disaster Economics                  16
     Beverages Industry                  15
     Sports Business                     13
     Corporate Earnings                  12
     Energy Markets                      11
     Housing Market                      11
     Pensi